# ElegyBox — Hierarchical Note Model Training

Run this on **Lightning AI (T4 GPU)**. Two-phase training:

| Phase | Epochs | What trains | LR |
|-------|--------|-------------|----|
| 1 | 15 | BarEncoder + cross-attention adapters only | 3e-4 |
| 2 | 20 | Everything jointly (old weights at tiny lr) | 5e-6 / 1e-4 |

### Architecture additions
- **`BarEncoder`** — small bidirectional transformer (d=128, 3 layers) that compresses each completed bar into 4 summary vectors projected to 384-dim.
- **`_BlockWithCrossAttn`** — drop-in for `_Block`; same `ln1/ln2/attn/ff` keys so existing weights load via `strict=False`; adds one cross-attention sub-layer.
- **`HierarchicalMusicGPT`** — note model with cross-attention adapters. When `memory=None` it behaves identically to the original.

Key transposition augmentation: ±6 semitones, applied consistently across history, prefix, and note tokens.

## Before you run

1. **Local machine** — generate the hierarchical dataset:
   ```
   python preprocess_hierarchical.py
   ```

2. **Upload into the same folder as this notebook** (same place you uploaded `note_samples.pkl.gz` before):
   - `note_model.pt`
   - `note_samples_hierarchical.pkl`

3. Run cells **top to bottom**.

4. **Cell 11** gives download links for `note_model_hierarchical.pt` and `bar_encoder.pt`. Place both in your local `ElegyBox/checkpoints/`.

## Cell 1 — Install dependencies

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'mido', 'tqdm', '-q'], check=True)
print('Dependencies ready')
print('Upload note_model.pt and note_samples_hierarchical.pkl into the same folder as this notebook.')

## Cell 2 — GPU check & paths

In [ ]:
import os, math, random, time, pickle, sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    torch.backends.cudnn.benchmark = True
else:
    print('No GPU found — training will be very slow.')

# All paths relative to the notebook's working directory
NOTE_CKPT = Path('note_model.pt')
HIER_DATA = Path('note_samples_hierarchical.pkl')

# Save hierarchical checkpoints to home dir — writable on all Lightning AI instances
_HOME     = Path.home()
HIER_CKPT = _HOME / 'note_model_hierarchical.pt'
ENC_CKPT  = _HOME / 'bar_encoder.pt'

# ── Tokenizer constants (mirrors tokenizer.py) ───────────────────────────────
NOTE_PAD      = 0
NOTE_BAR_END  = 1
NOTE_ROOT_OFF = 2
NOTE_QUAL_OFF = 14
NOTE_NONE     = 24
NOTE_POS_OFF  = 25
NOTE_ON_OFF   = 41
NOTE_DUR_OFF  = 129
NOTE_VEL_OFF  = 145
NOTE_VOCAB    = 153

_NOTE_ON_END  = NOTE_ON_OFF + 88
_N_ROOTS      = 12

print(f'Working dir : {Path.cwd()}')
print(f'Home dir    : {_HOME}')
print(f'Base model  : {NOTE_CKPT}')
print(f'Hier model  : {HIER_CKPT}  (exists={HIER_CKPT.exists()})')
print(f'Bar encoder : {ENC_CKPT}  (exists={ENC_CKPT.exists()})')

## Cell 3 — Configuration

Edit these values if needed. Defaults are tuned for a T4 (16 GB).

In [ ]:
# ── Bar encoder ───────────────────────────────────────────────────────────────
N_SUMMARY    = 4
ENC_D_MODEL  = 128
ENC_N_HEADS  = 4
ENC_N_LAYERS = 3
ENC_MAX_LEN  = 200
NOTE_D_MODEL = 384

# ── History window ────────────────────────────────────────────────────────────
MAX_HISTORY  = 16

# ── Augmentation ──────────────────────────────────────────────────────────────
MAX_SHIFT    = 6

# ── DataLoader ────────────────────────────────────────────────────────────────
BATCH_SIZE   = 64     # doubled — T4 has headroom at seq_len=320
NUM_WORKERS  = 4
VAL_SPLIT    = 0.05
GRAD_CLIP    = 1.0
SEQ_LEN      = 320

# ── Phase 1 ───────────────────────────────────────────────────────────────────
PHASE1_EPOCHS = 15
PHASE1_LR     = 3e-4

# ── Phase 2 ───────────────────────────────────────────────────────────────────
PHASE2_EPOCHS = 20
PHASE2_LR_NEW = 1e-4
PHASE2_LR_OLD = 5e-6

print('Config OK')

## Cell 4 — Architecture

**`BarEncoder`** — bidirectional transformer with learnable CLS-style summary tokens.  
**`HierarchicalMusicGPT`** — note model with cross-attention adapters. Existing `ln1/ln2/attn/ff` weights load from `note_model.pt` via `strict=False`; the new `ln_cross/cross_attn` keys are randomly initialised.

In [ ]:
# ── BarEncoder ────────────────────────────────────────────────────────────────

class _BidirBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout,
                                          batch_first=True)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout),
        )

    def forward(self, x):
        h = self.ln1(x)
        h, _ = self.attn(h, h, h, need_weights=False)  # no causal mask
        x = x + h
        x = x + self.ff(self.ln2(x))
        return x


class BarEncoder(nn.Module):
    """Compresses a completed bar into N_SUMMARY fixed vectors (~1.5 M params)."""
    def __init__(self, vocab_size=NOTE_VOCAB, d_enc=ENC_D_MODEL,
                 n_heads=ENC_N_HEADS, n_layers=ENC_N_LAYERS,
                 max_len=ENC_MAX_LEN, n_summary=N_SUMMARY,
                 note_d_model=NOTE_D_MODEL):
        super().__init__()
        self.n_summary = n_summary
        self.max_len   = max_len
        self.tok_emb   = nn.Embedding(vocab_size, d_enc)
        self.pos_emb   = nn.Embedding(max_len + n_summary, d_enc)
        self.summary   = nn.Parameter(torch.zeros(1, n_summary, d_enc))
        nn.init.normal_(self.summary, std=0.02)
        self.blocks    = nn.ModuleList([_BidirBlock(d_enc, n_heads) for _ in range(n_layers)])
        self.ln_f      = nn.LayerNorm(d_enc)
        self.proj      = nn.Linear(d_enc, note_d_model)

    def forward(self, bar_tokens):
        """bar_tokens: (B, T) -> (B, n_summary, note_d_model)"""
        B, T = bar_tokens.shape
        T    = min(T, self.max_len)
        x    = self.tok_emb(bar_tokens[:, :T])
        summ = self.summary.expand(B, -1, -1)
        x    = torch.cat([summ, x], dim=1)
        x    = x + self.pos_emb(torch.arange(x.size(1), device=x.device).unsqueeze(0))
        for blk in self.blocks:
            x = blk(x)
        return self.proj(self.ln_f(x)[:, :self.n_summary])


# ── HierarchicalMusicGPT ──────────────────────────────────────────────────────

class _BlockWithCrossAttn(nn.Module):
    """Same ln1/ln2/attn/ff keys as models._Block — loads from existing checkpoint."""
    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.ln2  = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout,
                                          batch_first=True)
        self.ff   = nn.Sequential(
            nn.Linear(d_model, d_model * 4), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model), nn.Dropout(dropout),
        )
        self.ln_cross   = nn.LayerNorm(d_model)
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout,
                                                batch_first=True)

    def forward(self, x, causal_mask, memory=None, memory_key_mask=None):
        h = self.ln1(x)
        h, _ = self.attn(h, h, h, attn_mask=causal_mask, need_weights=False)
        x = x + h
        if memory is not None:
            h = self.ln_cross(x)
            h, _ = self.cross_attn(h, memory, memory,
                                   key_padding_mask=memory_key_mask,
                                   need_weights=False)
            x = x + h
        x = x + self.ff(self.ln2(x))
        return x


class HierarchicalMusicGPT(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, seq_len, dropout=0.1):
        super().__init__()
        self.seq_len = seq_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(seq_len, d_model)
        self.drop    = nn.Dropout(dropout)
        self.blocks  = nn.ModuleList(
            [_BlockWithCrossAttn(d_model, n_heads, dropout) for _ in range(n_layers)]
        )
        self.ln_f    = nn.LayerNorm(d_model)
        self.head    = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight
        causal = torch.triu(torch.full((seq_len, seq_len), float('-inf')), diagonal=1)
        self.register_buffer('causal_mask', causal, persistent=False)

    def forward(self, x, targets=None, loss_mask=None, memory=None, memory_key_mask=None):
        B, T = x.shape
        pos  = torch.arange(T, device=x.device).unsqueeze(0)
        h    = self.drop(self.tok_emb(x) + self.pos_emb(pos))
        mask = self.causal_mask[:T, :T]
        for blk in self.blocks:
            h = blk(h, mask, memory=memory, memory_key_mask=memory_key_mask)
        logits = self.head(self.ln_f(h))
        if targets is None:
            return logits
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)),
                               targets.reshape(-1), ignore_index=0, reduction='none')
        if loss_mask is not None:
            denom = loss_mask.reshape(-1).sum().clamp(min=1)
            loss  = (loss * loss_mask.reshape(-1)).sum() / denom
        else:
            loss = loss[(targets != 0).reshape(-1)].mean()
        return logits, loss


def make_hierarchical_note_model():
    return HierarchicalMusicGPT(
        vocab_size=153, d_model=384, n_heads=8, n_layers=8, seq_len=320, dropout=0.1
    )


print('Architecture defined')
print(f'  BarEncoder          : ~{sum(p.numel() for p in BarEncoder().parameters())/1e6:.1f} M params')
print(f'  HierarchicalMusicGPT: ~{sum(p.numel() for p in make_hierarchical_note_model().parameters())/1e6:.0f} M params')

## Cell 5 — Load data

Make sure `note_samples_hierarchical.pkl` is at `ElegyBox/data/processed/` before running.

In [ ]:
if not HIER_DATA.exists():
    raise FileNotFoundError(
        f'Missing: {HIER_DATA}\n\n'
        'Generate it locally with:\n'
        '    python preprocess_hierarchical.py\n\n'
        'then upload note_samples_hierarchical.pkl next to this notebook.'
    )

print(f'Loading {HIER_DATA} ...')
with open(HIER_DATA, 'rb') as f:
    all_samples = pickle.load(f)
print(f'  {len(all_samples):,} samples loaded')

h, p, n = all_samples[100]
print(f'  Sample[100]: history={len(h)} bars, prefix={len(p)} tok, notes={len(n)} tok')


# ── Augmentation helper ───────────────────────────────────────────────────────
def _transpose_token_list(tokens, semitones):
    if semitones == 0:
        return tokens
    result = []
    for tok in tokens:
        if NOTE_ON_OFF <= tok < _NOTE_ON_END:
            new_idx = max(0, min(_NOTE_ON_END - NOTE_ON_OFF - 1,
                                 (tok - NOTE_ON_OFF) + semitones))
            result.append(NOTE_ON_OFF + new_idx)
        elif NOTE_ROOT_OFF <= tok < NOTE_ROOT_OFF + _N_ROOTS:
            result.append(NOTE_ROOT_OFF + (tok - NOTE_ROOT_OFF + semitones) % _N_ROOTS)
        else:
            result.append(tok)
    return result


# ── Dataset ───────────────────────────────────────────────────────────────────
class HierarchicalNoteDataset(Dataset):
    def __init__(self, samples, seq_len=SEQ_LEN, augment=True,
                 max_shift=MAX_SHIFT, max_history=MAX_HISTORY):
        self.seq_len     = seq_len
        self.augment     = augment
        self.max_shift   = max_shift
        self.max_history = max_history
        self.samples     = [s for s in samples
                            if len(s[1]) + len(s[2]) <= seq_len + 1]

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        history, prefix, notes = self.samples[idx]
        history = [list(h) for h in history]
        prefix  = list(prefix)
        notes   = list(notes)

        if self.augment and self.max_shift > 0:
            shift = random.randint(-self.max_shift, self.max_shift)
            if shift != 0:
                history = [_transpose_token_list(h, shift) for h in history]
                prefix  = _transpose_token_list(prefix, shift)
                notes   = _transpose_token_list(notes,  shift)

        history = history[-self.max_history:]

        full = prefix + notes
        need = self.seq_len + 1
        if len(full) < need:
            full = full + [NOTE_PAD] * (need - len(full))
        full = full[:need]

        mask       = [0.0] * len(full)
        note_start = len(prefix) - 1
        for j in range(note_start, len(mask)):
            mask[j] = 1.0

        x       = torch.tensor(full[:-1], dtype=torch.long)
        targets = torch.tensor(full[1:],  dtype=torch.long)
        lmask   = torch.tensor(mask[1:],  dtype=torch.float)
        lmask[targets == NOTE_PAD] = 0.0

        return x, targets, lmask, history   # history: list of token lists


def hierarchical_collate(batch):
    """
    Pre-pads bar histories into a (B, MAX_HISTORY, max_bar_len) tensor so
    encode_histories can do a single flat forward pass with no Python loops.
    """
    xs, targets, masks, histories = zip(*batch)

    # Pad sequences
    max_len = max(x.size(0) for x in xs)
    B       = len(xs)
    x_pad   = torch.zeros(B, max_len, dtype=torch.long)
    t_pad   = torch.zeros(B, max_len, dtype=torch.long)
    m_pad   = torch.zeros(B, max_len, dtype=torch.float)
    for i, (x, t, m) in enumerate(zip(xs, targets, masks)):
        L = x.size(0)
        x_pad[i, :L] = x
        t_pad[i, :L] = t
        m_pad[i, :L] = m

    # Pre-pad bar histories → (B, MAX_HISTORY, max_bar_len)
    all_bars    = [bar for h in histories for bar in h[-MAX_HISTORY:]]
    max_bar_len = min(max((len(b) for b in all_bars), default=1), ENC_MAX_LEN)

    hist_pad  = torch.zeros(B, MAX_HISTORY, max_bar_len, dtype=torch.long)
    hist_mask = torch.ones(B, MAX_HISTORY, dtype=torch.bool)   # True = ignore

    for i, h in enumerate(histories):
        h = h[-MAX_HISTORY:]
        for j, bar in enumerate(h):
            bar = bar[:max_bar_len]
            hist_pad[i, j, :len(bar)] = torch.tensor(bar, dtype=torch.long)
            hist_mask[i, j] = False   # valid bar

    return x_pad, t_pad, m_pad, hist_pad, hist_mask


# ── Train / val split ─────────────────────────────────────────────────────────
random.seed(42)
shuffled = list(all_samples)
random.shuffle(shuffled)
n_val    = max(1, int(len(shuffled) * VAL_SPLIT))
train_ds = HierarchicalNoteDataset(shuffled[n_val:],  augment=True)
val_ds   = HierarchicalNoteDataset(shuffled[:n_val],  augment=False)

loader_kw    = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                    pin_memory=True, persistent_workers=True,
                    collate_fn=hierarchical_collate)
train_loader = DataLoader(train_ds, shuffle=True,  drop_last=True,  **loader_kw)
val_loader   = DataLoader(val_ds,   shuffle=False, drop_last=False, **loader_kw)

print(f'Train: {len(train_ds):,}   Val: {len(val_ds):,}')
print(f'Batches per epoch: {len(train_loader):,}')

## Cell 6 — Training utilities

In [ ]:
def encode_histories(bar_encoder, hist_pad, hist_mask, device, n_summary=N_SUMMARY):
    """
    Single flat forward pass through the bar encoder — no Python loops on the hot path.

    hist_pad  : (B, H, T)  pre-padded bar tokens from collate
    hist_mask : (B, H)     True = no bar (padding slot)
    Returns memory (B, H*n_summary, D), mem_key_mask (B, H*n_summary)
    """
    B, H, T = hist_pad.shape
    D       = bar_encoder.proj.out_features

    hist_pad  = hist_pad.to(device, non_blocking=True)
    hist_mask = hist_mask.to(device, non_blocking=True)

    # Flatten all bars into one batch, encode, reshape back
    embs = bar_encoder(hist_pad.reshape(B * H, T))   # (B*H, n_summary, D)
    embs = embs.reshape(B, H, n_summary, D)

    memory       = embs.reshape(B, H * n_summary, D)
    mem_key_mask = hist_mask.unsqueeze(-1).expand(B, H, n_summary).reshape(B, H * n_summary)

    return memory, mem_key_mask


def cosine_schedule(optimizer, warmup_steps, total_steps):
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        t = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.05, 0.5 * (1.0 + math.cos(math.pi * t)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def run_epoch(note_model, bar_encoder, loader, optimizer, scaler,
              scheduler, device, train=True):
    note_model.train(train)
    bar_encoder.train(train)
    total_loss, n = 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, targets, loss_mask, hist_pad, hist_mask in loader:
            x         = x.to(device, non_blocking=True)
            targets   = targets.to(device, non_blocking=True)
            loss_mask = loss_mask.to(device, non_blocking=True)

            if train:
                optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
                memory, mem_key_mask = encode_histories(
                    bar_encoder, hist_pad, hist_mask, device)
                _, loss = note_model(x, targets, loss_mask,
                                     memory=memory, memory_key_mask=mem_key_mask)

            if train:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(
                    list(note_model.parameters()) + list(bar_encoder.parameters()),
                    GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

            total_loss += loss.item()
            n          += 1

    return total_loss / max(n, 1)


def save_checkpoint(note_model, bar_encoder, epoch, val_loss):
    # Unwrap torch.compile wrapper for saving
    raw_note = getattr(note_model, '_orig_mod', note_model)
    raw_enc  = getattr(bar_encoder, '_orig_mod', bar_encoder)
    torch.save({
        'epoch': epoch, 'val_loss': val_loss,
        'config': dict(vocab_size=153, d_model=384, n_heads=8,
                       n_layers=8, seq_len=320, dropout=0.1),
        'state': raw_note.state_dict(),
    }, HIER_CKPT)
    torch.save({
        'epoch': epoch, 'val_loss': val_loss,
        'config': dict(vocab_size=NOTE_VOCAB, d_enc=ENC_D_MODEL,
                       n_heads=ENC_N_HEADS, n_layers=ENC_N_LAYERS,
                       max_len=ENC_MAX_LEN, n_summary=N_SUMMARY,
                       note_d_model=NOTE_D_MODEL),
        'state': raw_enc.state_dict(),
    }, ENC_CKPT)
    print(f'  Saved  epoch={epoch}  val={val_loss:.4f}')


print('Utilities ready')

## Cell 7 — Instantiate models & load epoch-50 weights

Existing `ln1/ln2/attn/ff` keys load cleanly. The 32 new cross-attention keys (`ln_cross`, `cross_attn`) are randomly initialised — that's expected.

In [ ]:
bar_encoder = BarEncoder().to(device)
note_model  = make_hierarchical_note_model().to(device)

if HIER_CKPT.exists() and ENC_CKPT.exists():
    # Resume from a saved hierarchical checkpoint (e.g. after Phase 1)
    ckpt     = torch.load(HIER_CKPT, map_location=device, weights_only=False)
    enc_ckpt = torch.load(ENC_CKPT,  map_location=device, weights_only=False)
    note_model.load_state_dict(ckpt['state'])
    bar_encoder.load_state_dict(enc_ckpt['state'])
    print(f'Resumed hierarchical checkpoint: epoch={ckpt.get("epoch","?")}  val={ckpt.get("val_loss",0):.4f}')
else:
    assert NOTE_CKPT.exists(), f'Checkpoint not found: {NOTE_CKPT}'
    ckpt = torch.load(NOTE_CKPT, map_location=device, weights_only=False)
    missing, unexpected = note_model.load_state_dict(ckpt['state'], strict=False)
    print(f'Loaded base note model from {NOTE_CKPT.name}')
    print(f'  Missing keys  (new cross-attn) : {len(missing)}')
    print(f'  Unexpected keys (should be 0)  : {len(unexpected)}')
    bad = [k for k in missing if 'cross_attn' not in k and 'ln_cross' not in k]
    assert not bad, f'Unexpected missing keys: {bad}'
    print(f'  All {len(missing)} missing keys are cross-attention adapter weights  OK')

n_note = sum(p.numel() for p in note_model.parameters())
n_enc  = sum(p.numel() for p in bar_encoder.parameters())
print(f'\nNote model : {n_note/1e6:.1f} M params')
print(f'Bar encoder: {n_enc/1e6:.1f} M params')

if hasattr(torch, 'compile'):
    note_model  = torch.compile(note_model)
    bar_encoder = torch.compile(bar_encoder)
    print('\ntorch.compile enabled (first epoch will be slower while JIT-compiling)')

## Cell 8 — Phase 1: train adapters + bar encoder only

The existing note model weights are **frozen**. Only the 32 new cross-attention keys and the bar encoder are updated. This lets the encoder learn to produce useful bar summaries before the note model has to adapt to them.

In [ ]:
# ── Skip Phase 1 if a hierarchical checkpoint already exists ──────────────────
if HIER_CKPT.exists() and ENC_CKPT.exists():
    ckpt     = torch.load(HIER_CKPT, map_location='cpu', weights_only=False)
    best_val = ckpt.get('val_loss', float('inf'))
    history  = []
    print(f'Phase 1 already complete — epoch={ckpt.get("epoch","?")}  best_val={best_val:.4f}')
    print('Skipping Phase 1 training. Proceed to Cell 9 (Phase 2).')
else:
    # Freeze everything except the cross-attention adapters
    for param in note_model.parameters():
        param.requires_grad = False
    for name, param in note_model.named_parameters():
        if 'cross_attn' in name or 'ln_cross' in name:
            param.requires_grad = True

    trainable_adapters = sum(p.numel() for p in note_model.parameters() if p.requires_grad)
    trainable_encoder  = sum(p.numel() for p in bar_encoder.parameters())
    print(f'Phase 1 — trainable params')
    print(f'  Cross-attn adapters : {trainable_adapters:,}')
    print(f'  Bar encoder         : {trainable_encoder:,}')

    p1_params   = ([p for p in note_model.parameters() if p.requires_grad]
                   + list(bar_encoder.parameters()))
    optimizer   = torch.optim.AdamW(p1_params, lr=PHASE1_LR, weight_decay=0.01)
    total_steps = PHASE1_EPOCHS * len(train_loader)
    warmup      = min(500, total_steps // 10)
    scheduler   = cosine_schedule(optimizer, warmup, total_steps)
    scaler      = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

    best_val = float('inf')
    history  = []
    epoch_bar = tqdm(range(1, PHASE1_EPOCHS + 1), desc='Phase 1')

    for epoch in epoch_bar:
        t0  = time.time()
        tr  = run_epoch(note_model, bar_encoder, train_loader,
                        optimizer, scaler, scheduler, device, train=True)
        val = run_epoch(note_model, bar_encoder, val_loader,
                        optimizer, scaler, scheduler, device, train=False)
        history.append(('P1', epoch, tr, val))

        marker = ''
        if val < best_val:
            best_val = val
            save_checkpoint(note_model, bar_encoder, epoch, val)
            marker = '  <- saved'

        epoch_bar.set_postfix(train=f'{tr:.4f}', val=f'{val:.4f}')
        print(f'  P1 [{epoch:2d}/{PHASE1_EPOCHS}]  '
              f'train={tr:.4f}  val={val:.4f}  '
              f'lr={scheduler.get_last_lr()[0]:.2e}  {time.time()-t0:.0f}s{marker}')

    print(f'\nPhase 1 complete.  Best val: {best_val:.4f}')

## Cell 9 — Phase 2: full joint fine-tune

Two learning-rate groups:
- **Existing note model weights** → `5e-6` (very conservative — preserve what was learned at epoch 50)
- **Cross-attention adapters + bar encoder** → `1e-4` (continued learning)

In [ ]:
# Recreate DataLoaders — persistent workers from Phase 1 die between phases
loader_kw    = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                    pin_memory=True, persistent_workers=True,
                    collate_fn=hierarchical_collate)
train_loader = DataLoader(train_ds, shuffle=True,  drop_last=True,  **loader_kw)
val_loader   = DataLoader(val_ds,   shuffle=False, drop_last=False, **loader_kw)
print(f'DataLoaders ready  (train={len(train_loader)} batches, val={len(val_loader)} batches)')

# Guard: if Cell 8 was skipped, initialize history and best_val from checkpoint
if 'history' not in dir():
    history = []
if 'best_val' not in dir():
    if HIER_CKPT.exists():
        _ckpt    = torch.load(HIER_CKPT, map_location='cpu', weights_only=False)
        best_val = _ckpt.get('val_loss', float('inf'))
        print(f'best_val loaded from checkpoint: {best_val:.4f}')
    else:
        best_val = float('inf')

for param in note_model.parameters():
    param.requires_grad = True

old_params = [p for name, p in note_model.named_parameters()
              if 'cross_attn' not in name and 'ln_cross' not in name]
new_params = ([p for name, p in note_model.named_parameters()
               if 'cross_attn' in name or 'ln_cross' in name]
              + list(bar_encoder.parameters()))

print('Phase 2 — param groups')
print(f'  Old note model  (lr={PHASE2_LR_OLD}) : {sum(p.numel() for p in old_params):>10,}')
print(f'  Adapters+encoder(lr={PHASE2_LR_NEW}) : {sum(p.numel() for p in new_params):>10,}')

optimizer = torch.optim.AdamW(
    [{'params': old_params, 'lr': PHASE2_LR_OLD},
     {'params': new_params, 'lr': PHASE2_LR_NEW}],
    weight_decay=0.01,
)
total_steps = PHASE2_EPOCHS * len(train_loader)
warmup      = min(300, total_steps // 10)
scheduler   = cosine_schedule(optimizer, warmup, total_steps)
scaler      = torch.amp.GradScaler(enabled=(device.type == 'cuda'))

epoch_bar = tqdm(range(1, PHASE2_EPOCHS + 1), desc='Phase 2')

for epoch in epoch_bar:
    t0  = time.time()
    tr  = run_epoch(note_model, bar_encoder, train_loader,
                    optimizer, scaler, scheduler, device, train=True)
    val = run_epoch(note_model, bar_encoder, val_loader,
                    optimizer, scaler, scheduler, device, train=False)
    history.append(('P2', epoch, tr, val))

    marker = ''
    if val < best_val:
        best_val = val
        save_checkpoint(note_model, bar_encoder, epoch, val)
        marker = '  <- saved'

    epoch_bar.set_postfix(train=f'{tr:.4f}', val=f'{val:.4f}')
    print(f'  P2 [{epoch:2d}/{PHASE2_EPOCHS}]  '
          f'train={tr:.4f}  val={val:.4f}  '
          f'lr_old={optimizer.param_groups[0]["lr"]:.1e}  '
          f'lr_new={optimizer.param_groups[1]["lr"]:.1e}  '
          f'{time.time()-t0:.0f}s{marker}')

print(f'\nPhase 2 complete.  Best val overall: {best_val:.4f}')
print(f'Checkpoints: {HIER_CKPT.name}, {ENC_CKPT.name}')

## Cell 10 — Plot training curve (optional)

In [ ]:
try:
    import matplotlib.pyplot as plt
    tr_hist  = [h[2] for h in history]
    val_hist = [h[3] for h in history]
    phases   = [h[0] for h in history]

    p2_start = next((i for i, h in enumerate(history) if h[0] == 'P2'), len(history))

    plt.figure(figsize=(10, 4))
    plt.plot(tr_hist,  label='train loss')
    plt.plot(val_hist, label='val loss')
    if p2_start < len(history):
        plt.axvline(p2_start, color='grey', linestyle='--', alpha=0.6, label='Phase 2 start')
    plt.xlabel('Epoch (P1 then P2)')
    plt.ylabel('Loss')
    plt.title('Hierarchical Note Model Training')
    plt.legend()
    plt.tight_layout()
    plt.savefig('training_curve_hierarchical.png', dpi=120)
    plt.show()
    print('Saved training_curve_hierarchical.png')
except ImportError:
    print('matplotlib not installed — skipping plot')

## Cell 11 — Download checkpoints

Click the links to download both files. Place them in your local `ElegyBox/checkpoints/`.

In [ ]:
from IPython.display import FileLink, display

for path in [HIER_CKPT, ENC_CKPT]:
    size_mb = path.stat().st_size / 1024**2
    print(f'{path.name}  ({size_mb:.1f} MB)')
    display(FileLink(str(path)))
    print()

## Cell 12 — Next steps: updating generate.py for inference

After downloading the checkpoints, update `generate.py` as follows:

### 1. Imports
Add `HierarchicalMusicGPT`, `BarEncoder`, and `encode_histories` (copy the functions from this notebook, or factor them into a `models_hierarchical.py`).

### 2. Load both checkpoints
```python
note_ckpt   = torch.load('checkpoints/note_model_hierarchical.pt', map_location=device)
note_model  = HierarchicalMusicGPT(**note_ckpt['config']).to(device)
note_model.load_state_dict(note_ckpt['state'])
note_model.eval()

enc_ckpt    = torch.load('checkpoints/bar_encoder.pt', map_location=device)
bar_encoder = BarEncoder(**enc_ckpt['config']).to(device)
bar_encoder.load_state_dict(enc_ckpt['state'])
bar_encoder.eval()
```

### 3. Running bar history
Before the generation loop:
```python
bar_history = []   # list of raw note-token lists
```

### 4. Before each `generate_bar()` call
```python
if bar_history:
    memory, mem_key_mask = encode_histories(bar_encoder, [bar_history], device)
else:
    memory = mem_key_mask = None
```

### 5. Pass memory into `generate_bar()`
Update the function signature to accept and forward `memory` + `mem_key_mask` to `note_model()`.

### 6. After `generate_bar()` returns
```python
bar_history.append(raw_tokens)
if section_changed:
    bar_history = []   # reset at section boundaries
```

The chord model is unchanged.